# Izgradnja aplikacija za generiranje slika (OpenAI)

LLM-ovi nisu samo za generiranje teksta. Također možete generirati slike iz tekstualnih opisa. Slike kao modalitet korisne su u MedTech-u, arhitekturi, turizmu, razvoju igara, marketingu i drugim područjima. U ovoj lekciji radimo s današnjim **GPT Image** modelima na OpenAI platformi.

## Ciljevi učenja

Na kraju ove lekcije ćete moći:

- Objasniti što je generiranje slika i gdje je korisno.
- Razumjeti porodicu modela `gpt-image` i kako se razlikuje od naslijeđenih DALL·E modela.
- Izraditi aplikaciju za generiranje slika i uređivati slike.

## Što je generiranje slika?

Modeli za generiranje slika stvaraju slike na temelju tekstualnog upita. Moderni modeli poput `gpt-image` uče odnos između teksta i slika tijekom treniranja, a zatim iterativno pretvaraju slučajnu šumu u sliku koja odgovara vašem upitu.

Dvije dobro poznate obitelji modela za slike su:

- **`gpt-image` (OpenAI)** — trenutna generacija korištena u ovoj lekciji. Podržava generiranje slika iz teksta i uređivanje slika (inpainting s maskom).
- **Midjourney** — popularni model treće strane s vlastitom uslugom i radnim tokovima baziranim na Discord-u.

> Stariji OpenAI modeli za slike — **DALL·E 2** i **DALL·E 3** — su naslijeđeni modeli. DALL·E 3 više nije dostupan za nove implementacije, a značajke poput `create_variation` postojale su samo u DALL·E 2. Za nove aplikacije koristite modele `gpt-image`.

> **Važno:** `gpt-image` modeli vraćaju generiranu sliku kao **base64** (`b64_json`), a ne kao URL. Vaš kôd dekodira base64 niz u bajtove i sprema ih — ne postoji URL slike za preuzimanje.


## Izrada vaše prve aplikacije za generiranje slika

Pa što je potrebno za izradu aplikacije za generiranje slika? Potrebne su vam sljedeće biblioteke:

- **python-dotenv**, snažno se preporučuje korištenje ove biblioteke za čuvanje vaših tajni u *.env* datoteci, odvojeno od koda.
- **openai**, ovu biblioteku ćete koristiti za interakciju s OpenAI API-jem.
- **pillow**, za rad sa slikama u Pythonu.


1. Izradite datoteku *.env* s sljedećim sadržajem:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Prikupite gore navedene biblioteke u datoteku nazvanu *requirements.txt* na sljedeći način:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Zatim, kreirajte virtualno okruženje i instalirajte biblioteke:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Za Windows koristite sljedeće naredbe za stvaranje i aktivaciju vašeg virtualnog okruženja:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Dodajte sljedeći kod u datoteku nazvanu *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Objasnimo ovaj kod:

- Prvo uvozimo potrebne biblioteke, uključujući OpenAI biblioteku, dotenv biblioteku, base64 modul i Pillow biblioteku.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Nakon toga, stvaramo klijenta, koji čita API ključ iz vaše ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Zatim generiramo sliku:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` modeli vraćaju sliku kao **base64** niz u `data[0].b64_json`. Dekodiramo je u bajtove i zapisujemo u datoteku — nema URL-a za preuzimanje.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Na kraju, otvaramo sliku i koristimo standardni preglednik slika da je prikažemo:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Više detalja o generiranju slike

Pogledajmo parametre `images.generate`:

- **model**, je model slike koji se koristi (na primjer `gpt-image-2`).
- **prompt**, je tekstualni upit koji se koristi za generiranje slike. Ovdje je to "Zeko na konju, držeći lizalicu, na maglovitoj livadi gdje rastu narcisi".
- **size**, je veličina generirane slike (na primjer `1024x1024`, `1536x1024`, `1024x1536` ili `"auto"`).
- **n**, je broj generiranih slika. Ovdje generiramo jednu.

> Modeli za slike ne koriste parametar `temperature` — to je kontrola generiranja teksta. Za raznolikost, pozovite API ponovno; za manje raznolikosti, učinite vaš upit specifičnijim.

## Dodatne mogućnosti generiranja slike

Vidjeli ste kako generirati sliku s nekoliko redaka Pythona. `gpt-image` modeli također mogu **uređivati** postojeću sliku. Davanjem slike, opcionalne **maske** (koja označava područje za promjenu) i upita, možete promijeniti dio slike — na primjer, dodati šešir našem zeki.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Osnovna slika sadrži samo zeca; konačna slika ima šešir na zecu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
